In [4]:
import pandas as pd
import numpy as np
import sqlite3

# 1. Load raw CSV files
players = pd.read_csv('raw_players.csv')
matches = pd.read_csv('raw_matches.csv')
sessions = pd.read_csv('raw_sessions.csv')

# 2. Clean data
sessions['ping_ms'] = sessions['ping_ms'].apply(lambda x: np.nan if x < 0 or x >= 999 else x)
sessions['ping_ms'] = sessions['ping_ms'].fillna(sessions['ping_ms'].median())

merged = matches.merge(sessions, left_index=True, right_index=True)

# Corrected column name to match_outcome
matches['match_outcome'] = np.where(
    matches['match_outcome'].isna() & (merged['disconnected'] == 1),
    'Forfeit',
    matches['match_outcome'].fillna('Draw')
)

# 3. Save clean tables to SQLite database
conn = sqlite3.connect('gaming_data.db')
players.to_sql('players', conn, if_exists='replace', index=False)
matches.to_sql('matches', conn, if_exists='replace', index=False)
sessions.to_sql('sessions', conn, if_exists='replace', index=False)

# 4. Run sample query
df = pd.read_sql_query("SELECT * FROM players LIMIT 5;", conn)
print("SUCCESS: Data loaded into SQLite database!")
display(df)

conn.close()

SUCCESS: Data loaded into SQLite database!


,player_id,username,region,join_date
0,P_1000,jenniferlopez,OCE,2025-12-11
1,P_1001,pjarvis,NaN,2026-06-10
2,P_1002,poconnor,SA,2025-11-21
3,P_1003,wbarton,OCE,2025-11-17
4,P_1004,christianchavez,EU,2026-08-07


In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="darkgrid")
conn = sqlite3.connect("gaming_data.db")

# Read tables directly using pandas
players = pd.read_sql_query("SELECT * FROM players;", conn)
matches = pd.read_sql_query("SELECT * FROM matches;", conn)
sessions = pd.read_sql_query("SELECT * FROM sessions;", conn)
conn.close()

# Merge sessions with players on player_id, and combine with matches index-wise or on match_id
df_viz = sessions.merge(players, on="player_id", how="left")

# Attach match outcomes
outcome_col = "match_outcome" if "match_outcome" in matches.columns else "outcome"
if "match_id" in matches.columns and "match_id" in df_viz.columns:
    df_viz = df_viz.merge(matches[["match_id", outcome_col]], on="match_id", how="left")
else:
    # Fallback merge if match_id isn't in sessions
    df_viz[outcome_col] = matches[outcome_col].reindex(df_viz.index).values

# Plot 1: Ping Latency Distribution by Region
plt.figure(figsize=(8, 4))
sns.boxplot(data=df_viz, x="region", y="ping_ms", palette="Set2")
plt.title("Server Ping Latency by Region", fontweight="bold")
plt.xlabel("Region")
plt.ylabel("Ping (ms)")
plt.tight_layout()
plt.savefig("ping_distribution.png")
plt.show()

# Plot 2: Match Outcomes by Region
plt.figure(figsize=(8, 4))
sns.countplot(data=df_viz, x="region", hue=outcome_col, palette="viridis")
plt.title("Match Outcomes Breakdown by Region", fontweight="bold")
plt.xlabel("Region")
plt.ylabel("Count")
plt.legend(title="Outcome")
plt.tight_layout()
plt.savefig("match_outcomes.png")
plt.show()